# Bank Marketing Term Deposit Prediction - Classification Track

## Capstone Project - Review 1

**Objective:** Predict whether a client will subscribe to a term deposit (`y` = 'yes'/'no') based on demographic, financial, and campaign features.

* **Reproducibility:** Global `random_state=42` is strictly enforced across all splits, cross-validation folds, and stochastic estimators.
* **Data Leakage Prevention:** All transformations, scalers, and encoders are strictly fitted on the training split only.
* **Evaluation Metrics:** Evaluated across Accuracy, Precision, Recall, Weighted F1-score, Confusion Matrix, and ROC-AUC.

### Environment Configuration & Core Imports
This cell loads the necessary libraries for data analysis (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), and scikit-learn modules. It sets up plot styling, a dynamic data path resolver, and configures the global random seed (`42`) for full reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Set LOKY_MAX_CPU_COUNT to prevent Windows joblib wmic core-detection warning
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 4))

# Reproducibility seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

def get_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join('..', rel_path) if not rel_path.startswith('..') else rel_path,
        os.path.join('../..', rel_path.lstrip('./\\')),
        os.path.join('data', os.path.basename(rel_path)),
        os.path.join('../data', os.path.basename(rel_path)),
        os.path.join('../../data', os.path.basename(rel_path)),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return rel_path

print("Classification notebook initialized. Random state set to 42.")

## Phase 2: Exploratory Data Analysis (EDA)

In this phase, we perform an extensive audit of the Bank Marketing dataset:
1. **Data Ingestion & Integrity Audit:** Schema check, data types, null counts, and unknown category flags.
2. **Target Variable Imbalance Audit (`y`):** Exact breakdown of `'yes'` vs `'no'` subscribers and metric implications.
3. **Demographic & Client Profile Exploration:** Age, job categories, marital status, and education distributions vs conversion rate.
4. **Financial Health & Credit Indicators:** Balances, housing loans, personal loans, and credit default flags.
5. **Campaign Interaction Patterns:** Call duration, campaign contacts, and previous marketing outcomes (`poutcome`).
6. **Feature Intercorrelations:** Correlation matrix of numerical variables.

### Step 1: Load Dataset & Inspect Initial Structure
Load the Bank Marketing dataset from `data/bank-full.csv` using semicolon separator (`sep=';'`). Display dataset shape and inspect initial records.

In [ ]:
bank_df = pd.read_csv(get_data_path('data/bank-full.csv'), sep=';')

print(f"Dataset Shape: {bank_df.shape[0]} rows, {bank_df.shape[1]} columns")
bank_df.head()

### Step 2: Data Integrity, Schema Audit & Missingness Check
Inspect column schemas, data types, check for explicit NaN values, and calculate frequencies of implicit missingness indicated by `'unknown'` categorical entries.

In [ ]:
print("=== Column Schema & Data Types ===")
print(bank_df.info())

print("\n=== Explicit Null Values Check ===")
null_counts = bank_df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No explicit NaN / null values found.")

print("\n=== 'unknown' Category Audit ===")
cat_cols = bank_df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    unknown_count = (bank_df[col] == 'unknown').sum()
    if unknown_count > 0:
        pct = (unknown_count / len(bank_df)) * 100
        print(f"Feature '{col}': {unknown_count} 'unknown' records ({pct:.2f}%)")

### Step 3: Target Variable Class Imbalance Audit (`y`)
Examine the class distribution of term deposit subscriptions (`'yes'` vs `'no'`) using count plots and pie charts to identify class imbalance severity and metric implications.

In [ ]:
target_counts = bank_df['y'].value_counts()
target_pct = bank_df['y'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Countplot
sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0], palette=['#3274a1', '#e1812c'])
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 500, f"{v:,} ({target_pct.iloc[i]:.1f}%)", ha='center', fontweight='bold')
axes[0].set_title('Target Class Distribution: Term Deposit Subscription (y)')
axes[0].set_xlabel('Subscribed to Term Deposit?')
axes[0].set_ylabel('Record Count')

# Pie Chart
axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%', colors=['#3274a1', '#e1812c'], startangle=90, explode=(0, 0.1))
axes[1].set_title('Target Proportion (Severe Imbalance Flag)')

plt.tight_layout()
plt.show()

print(f"Majority ('no'): {target_counts['no']:,} ({target_pct['no']:.2f}%)")
print(f"Minority ('yes'): {target_counts['yes']:,} ({target_pct['yes']:.2f}%)")
print(f"Imbalance Ratio: {target_counts['no'] / target_counts['yes']:.2f} : 1")

### Step 4: Client Demographic Analysis
Analyze client age distributions by target outcome, and calculate subscription rates across job categories, marital status, and education levels.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Age Distribution by Target
sns.histplot(data=bank_df, x='age', hue='y', kde=True, ax=axes[0, 0], palette=['#3274a1', '#e1812c'], bins=35)
axes[0, 0].set_title('Age Distribution by Subscription Status')
axes[0, 0].set_xlabel('Age')

# Subscription Rate by Job
job_rate = bank_df.groupby('job')['y'].apply(lambda s: (s == 'yes').mean() * 100).sort_values(ascending=False)
sns.barplot(x=job_rate.values, y=job_rate.index, ax=axes[0, 1], palette='viridis')
axes[0, 1].set_title('Subscription Rate (%) by Job Title')
axes[0, 1].set_xlabel('Subscription Rate (%)')

# Subscription Rate by Marital Status
marital_rate = bank_df.groupby('marital')['y'].apply(lambda s: (s == 'yes').mean() * 100).reset_index()
sns.barplot(data=marital_rate, x='marital', y='y', ax=axes[1, 0], palette='Blues_r')
axes[1, 0].set_title('Subscription Rate (%) by Marital Status')
axes[1, 0].set_ylabel('Subscription Rate (%)')

# Subscription Rate by Education
edu_rate = bank_df.groupby('education')['y'].apply(lambda s: (s == 'yes').mean() * 100).reset_index()
sns.barplot(data=edu_rate, x='education', y='y', ax=axes[1, 1], palette='Purples_r')
axes[1, 1].set_title('Subscription Rate (%) by Education Level')
axes[1, 1].set_ylabel('Subscription Rate (%)')

plt.tight_layout()
plt.show()

### Step 5: Financial Health & Credit Indicators
Investigate yearly bank balances by subscription status, as well as conversion rates across housing loan and personal loan holders.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Balance Boxplot by Target (clipping extreme values for visualization clarity)
sns.boxplot(data=bank_df, x='y', y='balance', ax=axes[0], palette=['#3274a1', '#e1812c'], showfliers=False)
axes[0].set_title('Average Yearly Balance (€) by Outcome (Excluding Outliers)')
axes[0].set_xlabel('Subscribed (y)')
axes[0].set_ylabel('Balance (€)')

# Housing Loan vs Target
housing_rate = bank_df.groupby('housing')['y'].apply(lambda s: (s == 'yes').mean() * 100).reset_index()
sns.barplot(data=housing_rate, x='housing', y='y', ax=axes[1], palette='crest')
axes[1].set_title('Subscription Rate (%) by Housing Loan Status')
axes[1].set_ylabel('Subscription Rate (%)')

# Personal Loan vs Target
loan_rate = bank_df.groupby('loan')['y'].apply(lambda s: (s == 'yes').mean() * 100).reset_index()
sns.barplot(data=loan_rate, x='loan', y='y', ax=axes[2], palette='flare')
axes[2].set_title('Subscription Rate (%) by Personal Loan Status')
axes[2].set_ylabel('Subscription Rate (%)')

plt.tight_layout()
plt.show()

### Step 6: Campaign Interaction Patterns
Analyze call duration distributions, conversion rates across number of campaign contacts, and subscription rates by previous marketing outcome (`poutcome`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Duration vs Target (Log scale to handle high skewness)
sns.boxplot(data=bank_df, x='y', y='duration', ax=axes[0], palette=['#3274a1', '#e1812c'], showfliers=False)
axes[0].set_title('Call Duration (seconds) by Subscription Outcome')
axes[0].set_xlabel('Subscribed (y)')
axes[0].set_ylabel('Duration (sec)')

# Campaign Contacts vs Target (up to 10 contacts)
campaign_filtered = bank_df[bank_df['campaign'] <= 10]
camp_rate = campaign_filtered.groupby('campaign')['y'].apply(lambda s: (s == 'yes').mean() * 100).reset_index()
sns.lineplot(data=camp_rate, x='campaign', y='y', marker='o', ax=axes[1], color='#d95f02')
axes[1].set_title('Conversion Rate (%) vs Number of Campaign Contacts')
axes[1].set_xlabel('Number of Contacts During Campaign')
axes[1].set_ylabel('Subscription Rate (%)')

# Previous Campaign Outcome (poutcome)
poutcome_rate = bank_df.groupby('poutcome')['y'].apply(lambda s: (s == 'yes').mean() * 100).reset_index()
sns.barplot(data=poutcome_rate, x='poutcome', y='y', ax=axes[2], palette='magma')
axes[2].set_title('Subscription Rate (%) by Previous Outcome (poutcome)')
axes[2].set_xlabel('Previous Campaign Outcome')
axes[2].set_ylabel('Subscription Rate (%)')

plt.tight_layout()
plt.show()

### Step 7: Correlation Analysis for Numerical Variables
Compute and visualize pairwise Pearson correlation matrix among numerical attributes to screen for potential feature collinearity.

In [ ]:
num_cols = bank_df.select_dtypes(include=[np.number]).columns
corr_bank = bank_df[num_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_bank, dtype=bool))
sns.heatmap(corr_bank, annot=True, fmt='.2f', cmap='Blues', mask=mask, linewidths=0.5)
plt.title('Pearson Correlation Matrix (Bank Marketing Numerical Features)', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

### Phase 2 Key EDA Findings & Modeling Implications

1. **Substantial Class Imbalance:**
   * Only **11.7%** (5,289 records) of clients subscribed to a term deposit, compared to **88.3%** (39,922 records) non-subscribers (7.55 : 1 imbalance ratio).
   * Metric takeaway: Accuracy alone is deceptive (a dummy classifier achieves 88.3% accuracy). Evaluation must prioritize **Weighted F1-score**, **Precision**, and **Recall**.
2. **Impact of Call Duration:**
   * `duration` has the strongest empirical relationship with conversion: successful calls have significantly higher median duration.
   * *Note for modeling:* As highlighted in the UCI dataset documentation, duration is not known before a call is made and is heavily correlated with the outcome.
3. **Previous Campaign History (`poutcome`):**
   * Clients with a previous campaign outcome of 'success' exhibit an outstanding subscription rate of $\approx 64.7\%$, making historical interaction a high-priority predictive signal.
4. **Client Demographics & Debt Status:**
   * Students and retired individuals demonstrate the highest propensity to subscribe ($> 25\%$).
   * Clients without housing loans and personal loans subscribe at more than double the rate of clients with active loans.
5. **Handling of Missing / `'unknown'` Values:**
   * `poutcome` (81.7%), `contact` (28.8%), and `education` (4.1%) contain `'unknown'` strings rather than standard NaNs. In Phase 3, these will be handled as distinct categories during encoding.

---
# Phase 3: Preprocessing & Feature Engineering (@Abhisarg)

In this phase, we clean and prepare the Bank Marketing dataset for classification modeling:
1. **Missing Value Strategy:** Preserve `'unknown'` categories as distinct valid levels.
2. **Feature Engineering:** Derive `pdays_contacted`, `total_contacts`, `is_retired_or_student`, and `duration_min`.
3. **Binary & Target Encoding:** Map `default`, `housing`, `loan`, and `y` to 0/1 integers.
4. **Stratified Split:** 80/20 train-test split (`random_state=42`, `stratify=y`).
5. **Leakage-Free Scaling & Encoding:** `StandardScaler` and `OneHotEncoder` fitted exclusively on `X_train`.

### Step 1: Ingestion & Missing Values Strategy
Load dataset and confirm 0 explicit missing values. Preserved `'unknown'` categories as valid informative levels for `job`, `education`, `contact`, and `poutcome`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

bank_df = pd.read_csv(get_data_path('data/bank-full.csv'), sep=';')

print("=== Explicit NaN Audit ===")
print(bank_df.isnull().sum())

print("\nPreserving 'unknown' strings as informative categorical levels for job, education, contact, poutcome.")
bank_clean = bank_df.copy()

### Step 2: Feature Engineering (@Abhisarg)
Derive domain-specific features based on Phase 2 EDA findings:
* `pdays_contacted`: Binary flag indicating if client was previously contacted (`pdays != -1`).
* `total_contacts`: Sum of campaign and previous contacts (`campaign + previous`).
* `is_retired_or_student`: Binary indicator for student/retired job categories (high conversion propensity).
* `duration_min`: Call duration converted to minutes (`duration / 60.0`).

In [ ]:
bank_clean['pdays_contacted'] = (bank_clean['pdays'] != -1).astype(int)
bank_clean['total_contacts'] = bank_clean['campaign'] + bank_clean['previous']
bank_clean['is_retired_or_student'] = bank_clean['job'].isin(['retired', 'student']).astype(int)
bank_clean['duration_min'] = bank_clean['duration'] / 60.0

print("Engineered Features Added: ['pdays_contacted', 'total_contacts', 'is_retired_or_student', 'duration_min']")

### Step 3 & 4: Binary Mapping & Feature Matrix Definition
Convert binary categorical attributes (`default`, `housing`, `loan`) and target (`y`) from `'no'`/`'yes'` strings to `0`/`1` integers. Separate predictor set $X$ and target vector $y$, identifying numerical and categorical columns.

In [ ]:
binary_map = {'no': 0, 'yes': 1}
for binary_col in ['default', 'housing', 'loan']:
    bank_clean[binary_col] = bank_clean[binary_col].map(binary_map)

bank_clean['y'] = bank_clean['y'].map(binary_map)
print("Binary columns mapped to 0/1 integers: ['default', 'housing', 'loan', 'y']")

X_bank = bank_clean.drop(columns=['y'])
y_bank = bank_clean['y']

cat_cols_bank = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
num_cols_bank = [col for col in X_bank.columns if col not in cat_cols_bank]

### Step 5: Reproducible Stratified Train-Test Split
Partition data into 80% training and 20% testing sets using `random_state=42` and `stratify=y_bank` to preserve the 7.55:1 class balance ratio in both splits.

In [ ]:
X_train_bank, X_test_bank, y_train_bank, y_test_bank = train_test_split(
    X_bank, y_bank, test_size=0.20, random_state=RANDOM_STATE, stratify=y_bank
)

print(f"Stratified Train-Test Split Completed (random_state={RANDOM_STATE}):")
print(f"- X_train shape: {X_train_bank.shape}")
print(f"- X_test shape:  {X_test_bank.shape}")
print(f"- Train Positive Class Ratio (y=1): {y_train_bank.mean()*100:.2f}%")
print(f"- Test Positive Class Ratio (y=1):  {y_test_bank.mean()*100:.2f}%")

### Step 6: Leakage-Free Encoding & Scaling
Construct a `ColumnTransformer` applying `StandardScaler` to numerical columns and `OneHotEncoder(drop='first')` to categorical variables.

**Crucial Leakage Prevention:** Scalers and encoders are fitted **strictly on `X_train_bank`**, ensuring no test set distributions leak into training.

In [ ]:
preprocessor_bank = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_bank),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols_bank)
    ]
)

X_train_bank_scaled = preprocessor_bank.fit_transform(X_train_bank)
X_test_bank_scaled = preprocessor_bank.transform(X_test_bank)

cat_feature_names_bank = preprocessor_bank.named_transformers_['cat'].get_feature_names_out(cat_cols_bank)
all_feature_names_bank = num_cols_bank + list(cat_feature_names_bank)

print(f"Processed Training Feature Matrix (Scaled & Encoded): {X_train_bank_scaled.shape}")
print(f"Processed Testing Feature Matrix (Scaled & Encoded):  {X_test_bank_scaled.shape}")
print(f"Total Scaled/Encoded Features: {len(all_feature_names_bank)}")

---
# Phase 5: Classification Track - Modeling Part A (@Rohan)

**Author / Assignee:** @Rohan  
**Track:** Bank Marketing Term Deposit Prediction (Classification Part A)  

### Review 1 Scope & Objectives:
As specified in the Capstone Guidelines (**Section 3.2** & **Review 1 Rubric Section D**):
1. **Model Training:** Train the 5 required baseline classification algorithms on the preprocessed training set (`X_train_bank_scaled`, `y_train_bank`):
   - **Logistic Regression**: Linear baseline; interpret feature coefficients and odds ratios ($\\exp(\\beta)$).
   - **K-Nearest Neighbors (KNN)**: Instance-based classification; evaluate distance metric (Minkowski/Euclidean) on scaled space.
   - **Gaussian Naive Bayes**: Probabilistic baseline; evaluate conditional independence assumption.
   - **Decision Tree Classifier**: Non-linear tree; constrain depth to prevent overfitting and visualize tree structure.
   - **Support Vector Classifier (SVC)**: Maximum-margin hyperplane via RBF kernel on scaled feature space.
2. **Evaluation Metrics:** Compute and report Train Accuracy, Test Accuracy, Precision, Recall, **Weighted F1-score** (essential due to $7.55:1$ class imbalance), and **ROC-AUC**.
3. **Confusion Matrix Diagnostics:** Generate and visualize confusion matrices for all 5 algorithms.
4. **Preliminary Comparison Table:** Consolidate and rank all 5 models by performance metrics in a structured summary DataFrame.

### Evaluation Helper & Unified Training Pipeline
Define `evaluate_classification_model()`, which fits an estimator and computes mandatory evaluation metrics: Train Accuracy, Test Accuracy, Precision, Recall, Weighted F1-score, ROC-AUC, and Confusion Matrix. Benchmark all 5 Part-A baseline classifiers.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Dictionaries to store fitted models, predictions, probabilities, and evaluation artifacts
phase5_models = {}
phase5_predictions = {}
phase5_probabilities = {}
phase5_conf_matrices = {}
phase5_results = []

def evaluate_classification_model(name, model, X_tr, y_tr, X_te, y_te):
    """
    Fits an estimator and computes mandatory Capstone evaluation metrics:
    Train Accuracy, Test Accuracy, Precision, Recall, Weighted F1-Score, and ROC-AUC.
    """
    # 1. Fit model on training split
    model.fit(X_tr, y_tr)
    
    # 2. Predict class labels
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # 3. Decision scores / probabilities for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_te_scores = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        y_te_scores = model.decision_function(X_te)
    else:
        y_te_scores = y_te_pred
        
    # 4. Compute metrics
    tr_acc = accuracy_score(y_tr, y_tr_pred)
    te_acc = accuracy_score(y_te, y_te_pred)
    prec = precision_score(y_te, y_te_pred, average='weighted', zero_division=0)
    rec = recall_score(y_te, y_te_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_te, y_te_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_te, y_te_scores)
    cm = confusion_matrix(y_te, y_te_pred)
    
    # Store artifacts
    phase5_models[name] = model
    phase5_predictions[name] = y_te_pred
    phase5_probabilities[name] = y_te_scores
    phase5_conf_matrices[name] = cm
    
    metrics = {
        'Model': name,
        'Train Acc': round(tr_acc, 4),
        'Test Acc': round(te_acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'Weighted F1': round(f1_weighted, 4),
        'ROC-AUC': round(roc_auc, 4)
    }
    phase5_results[:] = [r for r in phase5_results if r['Model'] != name]
    phase5_results.append(metrics)
    print(f"[{name}] Done -> Test Acc: {te_acc:.4f} | Weighted F1: {f1_weighted:.4f} | ROC-AUC: {roc_auc:.4f}")
    return metrics

# --- 1. Logistic Regression ---
# Linear baseline classifier with L2 regularization
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
evaluate_classification_model("Logistic Regression", lr_model, X_train_bank_scaled, y_train_bank, X_test_bank_scaled, y_test_bank)

# --- 2. K-Nearest Neighbors ---
# Instance-based classifier using Minkowski (Euclidean) distance on normalized feature space
knn_model = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
evaluate_classification_model("K-Nearest Neighbors", knn_model, X_train_bank_scaled, y_train_bank, X_test_bank_scaled, y_test_bank)

# --- 3. Gaussian Naive Bayes ---
# Probabilistic classifier assuming conditional feature independence
gnb_model = GaussianNB()
evaluate_classification_model("Gaussian Naive Bayes", gnb_model, X_train_bank_scaled, y_train_bank, X_test_bank_scaled, y_test_bank)

# --- 4. Decision Tree Classifier ---
# Non-linear tree; constrained max_depth=5 to balance complexity and generalizability
dt_model = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
evaluate_classification_model("Decision Tree Classifier", dt_model, X_train_bank_scaled, y_train_bank, X_test_bank_scaled, y_test_bank)

# --- 5. Support Vector Classifier (SVC) ---
# Kernel-based maximum-margin classifier using RBF kernel on scaled feature space
svc_model = SVC(C=1.0, kernel='rbf', random_state=RANDOM_STATE, cache_size=1000, probability=True)
evaluate_classification_model("Support Vector Classifier (SVC)", svc_model, X_train_bank_scaled, y_train_bank, X_test_bank_scaled, y_test_bank)

### Model Diagnostics & Interpretations
1. **Logistic Regression Odds Ratios:** Exponentiating coefficients ($\\text{OR} = \\exp(\\beta)$) yields odds multipliers for term deposit subscription.
2. **Decision Tree Structure:** Visualizing the hierarchical decision paths and top split criteria.

In [ ]:
# --- Logistic Regression Odds Ratio Interpretation ---
lr_coefs = pd.DataFrame({
    'Feature': all_feature_names_bank,
    'Coefficient': lr_model.coef_[0],
    'Odds_Ratio': np.exp(lr_model.coef_[0])
}).sort_values(by='Coefficient', ascending=False)

print("=== Logistic Regression: Top 5 Predictors Increasing Subscription Odds ===")
print(lr_coefs.head(5).to_string(index=False))

print("\n=== Logistic Regression: Top 5 Predictors Decreasing Subscription Odds ===")
print(lr_coefs.tail(5).to_string(index=False))

# --- Decision Tree Visualization (Top 3 Depths) ---
plt.figure(figsize=(22, 9))
plot_tree(
    dt_model,
    max_depth=3,
    feature_names=all_feature_names_bank,
    class_names=['No (0)', 'Yes (1)'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree Classifier Structure (Top 3 Levels Visualized)", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

### Confusion Matrix Diagnostics (Rubric D2)
In an imbalanced classification context ($88.3\\%$ negative vs $11.7\\%$ positive), confusion matrices reveal whether high overall accuracy translates into successful positive detection (subscribers) or simply reflects majority-class bias.

In [ ]:
# --- Confusion Matrix Diagnostics (All 5 Baseline Models) ---
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

model_names = list(phase5_models.keys())
for idx, name in enumerate(model_names):
    cm = phase5_conf_matrices[name]
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
        cbar=False, annot_kws={'size': 13, 'weight': 'bold'}
    )
    axes[idx].set_title(f"{name}\nConfusion Matrix", fontsize=12, pad=10)
    axes[idx].set_xlabel("Predicted Label (0: No, 1: Yes)", fontsize=10)
    axes[idx].set_ylabel("True Label (0: No, 1: Yes)", fontsize=10)
    axes[idx].set_xticklabels(['No (0)', 'Yes (1)'])
    axes[idx].set_yticklabels(['No (0)', 'Yes (1)'])

# Hide unused 6th subplot in 2x3 grid
axes[5].axis('off')

plt.suptitle("Phase 5: Confusion Matrices for All 5 Part-A Classifiers", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Preliminary Comparison Table (Rubric D2)
The consolidated summary table below benchmarks all five Part-A baseline classifiers across all required evaluation metrics, ranked by **Weighted F1-score**.

In [ ]:
# --- Preliminary Comparison Table ---
phase5_df = pd.DataFrame(phase5_results).sort_values(by='Weighted F1', ascending=False).reset_index(drop=True)

print("=" * 88)
print("=== PRELIMINARY CLASSIFICATION COMPARISON TABLE (PART A - REVIEW 1) ===")
print("=" * 88)
print(phase5_df.to_string(index=False))

# Display styled DataFrame representation in Jupyter
phase5_df

### Phase 5 Key Findings & Capstone Review 1 Commentary

1. **Performance Summary Across Baseline Models:**
   - **Support Vector Classifier (SVC)** achieved the highest overall Test Accuracy (0.9040) and Weighted F1-score (0.8897) among the five baseline models.
   - **Logistic Regression** achieved a competitive Test Accuracy of 0.9014 and Weighted F1-score of 0.8880, while obtaining the highest ROC-AUC score (0.9055).
   - **Decision Tree Classifier (`max_depth=5`)** delivered competitive performance with a Test Accuracy of 0.8985 and Weighted F1-score of 0.8819, while providing an interpretable set of decision rules.
   - **K-Nearest Neighbors (KNN)** achieved a Test Accuracy of 0.8947 and Weighted F1-score of 0.8838 after applying the preprocessing and feature scaling pipeline.
   - **Gaussian Naive Bayes** produced the lowest overall Test Accuracy (0.8590) and Weighted F1-score (0.8668) among the five models.

2. **Metric Implications in the Imbalanced Setting:**
   - The test dataset is imbalanced, with approximately 88.3% of samples belonging to class `0` (`no`) and 11.7% belonging to class `1` (`yes`).
   - Therefore, Accuracy and Weighted F1-score should be interpreted together with class-specific Precision and Recall rather than being considered alone.
   - Gaussian Naive Bayes achieved the highest recall for the positive class (`yes`) at 0.5482, while SVC and Logistic Regression achieved the strongest overall Weighted F1 performance.
   - This shows that the model with the best overall performance is not necessarily the model with the best ability to identify the minority positive class.

3. **Transition to Review 2 (Classification Part B):**
   - The preprocessing pipeline, evaluation metrics, confusion matrices, and preliminary comparison established in Part A provide the foundation for evaluating the five additional classification algorithms in Part B: Random Forest, AdaBoost, Gradient Boosting, Bagging, and MLP Classifier.